Welcome to ***DAMN-Q***!
==========================

> **Validation note:** This tutorial demonstrates component wiring and event flow for a three-node repeater. The `00`/`11` counts printed at the end show Z-basis correlations, which a classical mixture can also produce. For end-to-end entanglement validation—including the exact joint |Φ+⟩⟨Φ+| endpoint state before measurement, separate X- and Z-basis checks, and the analytic noisy-memory fidelity comparison—see [`tests/test_memory.py`](./tests/test_memory.py) and run `python -m pytest`.

Get ready for a sci-fi setup, as we are going to use quantum memories and entanglement swapping protocol! It will be cash money of you to check how entanglement swapping, [quantum teleportation](https://en.wikipedia.org/wiki/Quantum_teleportation) with two entangled pairs, works before diving face first into this tutorial.

The setup we are going to check in this tutorial is as follows:

![The setup](./Source_a.png "The setup")

_____________________________________

**1. Importing the files** 

The [Scheduler](./Scheduler.py) contains the notion of time! We need that to keep the order of the actions based on their delays! Also, it helps run the simulator for a certain time. 

In [1]:
from Scheduler import*

Next, let's import the files that contain our quantum component classes.

In [2]:
from Measurer_memory_paired import*
from Measurer import*
from Pauli_corrector import*
from Source import*
from Entanglment_swapping_module import*
from Orchestrator import*
from Memory import*
from Auto_connector import*
from Classic_data_reader import*

___________________________
**2. Defining the components**

First we need our qubit sources. The first parameter is the *Id* of the object. The second one points to the delay before the qubits emition. All classes have a delay parameter. The last parameter is responsible for the type of emiting qubits. "*2*" is for Bell States and "*1*" is for unentangled qubits.

***Important note!***

The source requires a delay greater than 0.

In [3]:
source_a = Source(1,1,2)
source_b = Source(2,1,2)

Memories are up next. As memeories they have a capacity parameter. By default they are set on FIFO meaning the qubits that arrived first will leave first.

In [4]:
memory_a_1 = Quantum_memory(1,0,0,100)
memory_a_2 = Quantum_memory(2,0,0,100)
memory_b_1 = Quantum_memory(3,0,0,100)
memory_b_2 = Quantum_memory(4,0,0,100)

The entanglement swapping object requires an orchestrator.

In [5]:
es = Entaglment_swapping_module(1, 1)
orchestrator = Orchestrator(1,0)


And finally, we need the Pauli corrector and the measurers to provide us with the results. There are two types of measurers: the "*Memory_paired*" measurer, which is able to contact the memory and receive data from the entangled swapper, and the typical measurer, which receives the qubit and measures it. Please note that both can measure on an X and Z basis with some probability. To set them on an X or Z basis, we can set the X_basis probability (1-Z_basis probability) to "*1*" or "*0*" respectively.

In [6]:
pc = Pauli_corrector(1,0)
m_a = Measurer_memory_paired(1,0,0,0)
m_b = Measuring_Device(2,0,0,0)

The reader checks if the outputs match in this example.

In [7]:
reader = Classic_data_reader(1)

____________________________
**3. Connections**

Let's start connecting our sources and memories. The parameter that comes with the "*connect()*" function is the delay in the Link object that is automatically defined here. By default, the delay parameter is 0. Also, here we can set the probability of loss of the qubits in the link. The same goes for the probability of depolarisation in the link object.

In [8]:
connect(source_a.out_ch_1, memory_a_1.ch_in_q, 5)
connect(source_a.out_ch_2, memory_a_2.ch_in_q, 5)
connect(source_b.out_ch_1, memory_b_1.ch_in_q, 5)
connect(source_b.out_ch_2, memory_b_2.ch_in_q, 5)

Next, we connect the memories to the Pauli corrector and the measurers.

In [9]:
connect(memory_a_1.ch_out_q, m_a.ch_in_q)
connect(memory_b_2.ch_out_q, pc.ch_in_q)
connect(pc.ch_out_q, m_b.ch_in_q)

Here we connect the memories to the entanglement swapper object.

In [10]:
connect(memory_a_2.ch_out_q, es.ch_in_q_1)
connect(memory_b_1.ch_out_q, es.ch_in_q_2)

The memories have a classic output to inform the successful arrival of a qubit. Let's connect them to Orchestrator.

In [11]:
connect(memory_a_2.ch_out_c, orchestrator.ch_in_memory_1)
connect(memory_b_1.ch_out_c, orchestrator.ch_in_memory_2)

After assuring the coupling of two qubits. The orchestrator will order the memories to pass the qubits to the entanglement swapper.

In [12]:
connect(orchestrator.ch_out_memory_1, memory_a_2.ch_in_c)
connect(orchestrator.ch_out_memory_2, memory_b_1.ch_in_c)

The entanglement swapper needs to send the measurement results to the Pauli corrector and the measurer.

In [13]:
connect(es.ch_out_c_1, m_a.ch_in_es,5)
connect(es.ch_out_c_2, pc.ch_in_es,5)

The pair of memories connected to the same source are in contact with each other. They need to make sure both ebits of a Bell state arrived. If a qubit went missing before arriving at a memory, the other memory will discard the arrived ebit.

In [14]:
connect(memory_a_1.ch_out_paired_memory, memory_a_2.ch_in_paired_memory,5)
connect(memory_a_2.ch_out_paired_memory, memory_a_1.ch_in_paired_memory,5)
connect(memory_b_1.ch_out_paired_memory, memory_b_2.ch_in_paired_memory,5)
connect(memory_b_2.ch_out_paired_memory, memory_b_1.ch_in_paired_memory,5)

The memory-paird measurer will need to communicate with the memory, as its name suggests!

In [15]:
connect(memory_a_1.ch_out_c, m_a.ch_in_memory_c)
connect(m_a.ch_out_memory_c, memory_a_1.ch_in_c)

Also the PC communicates with the memory to receive qubits.

In [16]:
connect(memory_b_2.ch_out_c, pc.ch_in_memory)
connect(pc.ch_out_memory, memory_b_2.ch_in_c)

And finally, we need to connect the measurers to the reader.

In [17]:
connect(m_a.ch_out_result, reader.input_2)
connect(m_b.ch_out_result, reader.input_1)

And we should indicate the duration of the simulation. Keep in mind the scale of the time is completely arbitary, and it depends on your delays'. 
_____________________________
**4. Results**

Running this simulation will show the Z-basis measurement counts for "00", "01", "10" and "11".

> **Note:** The displayed `00`/`11` counts only show Z-basis correlations, which a classically correlated mixture can also produce. To verify true end-to-end entanglement—the exact joint |Φ+⟩⟨Φ+| endpoint state before measurement, separate X- and Z-basis runs, and the analytic storage-depolarization fidelity—see [`tests/test_memory.py`](./tests/test_memory.py) and run `python -m pytest`.

***Note!*** 

Please keep in mind this simulator can run for way more steps, like 10 million, but running it as a Jupiter notebook is far from optimal. [Test_entanglement_swapping](./Test_entanglement_swapping.py) contains the same code as this tutorial. Running that is a better idea as Jupiter is unreliable for larger simulations.

In [18]:
scheduler.run(100)

print(f" Number of qubits arrived in different states: 00:{reader.count_00}, 01:{reader.count_01}, 10:{reader.count_10}, 11:{reader.count_11}")


 Number of qubits arrived in different states: 00:51, 01:0, 10:0, 11:32


***Next [Tutorial](./Tutorial_lossy.ipynb) will cover lossy simulation***